# Scanvolver tutorial: from Falcons.jl pointings to GPU convolution

Generate one year of pointings for a single boresight detector with Falcons.jl, using a generator that produces monthly chunks. Retain only a few selected HEALPix RINGs in a `RingScanBuffer`, prepare small sky and beam harmonic inputs, convolve them on the GPU, and read `(I,Q,U)` maps from HDF5.

The defaults are `nside=8`, `lmax=16`, 1 Hz sampling, and three rings near the equator. Pointings cover a full year, while convolution processes only the selected rings. The low sampling rate and synthetic harmonic coefficients make this a usage example.

The default full-year workflow was executed successfully on an NVIDIA RTX A6000 with Julia 1.12.6 at commit `08a861f`, including GPU convolution, HDF5 readback, and plotting. All three output rings contained finite values. This confirms that the workflow runs; it does not establish numerical accuracy or the scientific validity of the toy inputs.

Without a GPU, input preparation and scan-buffer construction still run, while GPU-dependent cells explicitly skip.


## 1. Environment

Use Julia 1.12 and the `Project.toml` in this directory. See [notebook/README.md](README.md) for dependency setup and Julia kernel registration.

The GPU steps require an NVIDIA GPU with `CUDA.functional() == true`. A CPU-only computer can run the Jupyter interface as long as the selected Julia kernel runs on a GPU machine. One GPU is used by default.

The next cell expects the kernel working directory to be either the repository root or `notebook/`.


In [ ]:
using Pkg

notebook_dir = isfile(joinpath(pwd(), "tutorial.ipynb")) ?
    pwd() : joinpath(pwd(), "notebook")
isfile(joinpath(notebook_dir, "tutorial.ipynb")) ||
    error("Set the working directory to the repository root or notebook/.")
Pkg.activate(notebook_dir)

using Scanvolver
import Falcons
import CUDA
using HDF5: h5open
using Printf: @printf
using Plots

gpu_available = CUDA.functional()
@info "Execution environment" julia=VERSION falcons=Base.pkgversion(Falcons) gpu_available
if !gpu_available
    @info "CPU preparation cells can run. GPU convolution and HDF5 readback will be skipped."
end


## 2. Configuration

`target_rings` is a contiguous, 1-based range. Ring `2*nside` lies on the equator, so this example selects it and its two neighbors. Use `r:r` for a single ring.

A year is defined as 365.25 days and divided into 12 equal intervals, called "months" here. These are not calendar months. Boundaries are elapsed seconds from the start of the mission; time does not reset each month. Intervals are half-open, `[start, stop)`, so boundary samples are not counted twice.

`lmax` is the band limit of the packed inputs; `lstart:lstop` is the range actually computed. This example uses the same upper limit for both. The values of `κ` and `bnmax` are chosen for a small demonstration and do not guarantee accuracy for production runs.


In [ ]:
nside = 8
lmax = 16
target_rings = (2*nside - 1):(2*nside + 1)

mission_days = 365.25
sampling_rate_hz = 1.0
chunk_count = 12
compute_precision = Float32
device_ids = [0]

cp = ConvolutionParams(
    nside_output=nside,
    lstart=0,
    lstop=lmax,
    lmax=lmax,
    numofsky=1,
    numofbeam=1,
    bnmax=2,
    κ=4,
    HWP=false,
    noise=false,
)

mission_seconds = round(Int, mission_days * 24 * 60 * 60)
@assert mission_seconds % chunk_count == 0
chunk_seconds = mission_seconds ÷ chunk_count
@assert sampling_rate_hz > 0 && isinteger(chunk_seconds * sampling_rate_hz)
@assert 1 <= first(target_rings) <= last(target_rings) <= ring_count(cp)
@assert cp.lmax >= 3  # The input-conversion example below uses l=3.

time_windows = [
    ((i - 1) * chunk_seconds, i * chunk_seconds)
    for i in 1:chunk_count
]
@assert first(time_windows)[1] == 0
@assert last(time_windows)[2] == mission_seconds
@assert all(time_windows[i][2] == time_windows[i+1][1] for i in 1:chunk_count-1)

show_cp(cp)
@printf("mission: %.2f days, %d chunks, %d samples\n",
        mission_days, chunk_count, round(Int, mission_seconds * sampling_rate_hz))


## 3. Prepare sky and beam inputs on the CPU

Each input uses HEALPix/healpy packed storage for nonnegative modes, with shape `(3, (lmax+1)*(lmax+2)÷2)`. The three rows are `(T,E,B)`, and packed indices use Julia's 1-based indexing.

To run without downloading external data, generate deterministic, nonzero coefficients. These are synthetic sky and beam inputs, not a measured beam or a CMB realization for scientific analysis. The beam contains only modes with `n <= bnmax`. Coefficients at `m=0` are real, and E/B coefficients at `l<2` are zero. The inputs are treated as being in the same ecliptic coordinate system as the pointings below.

Replace `raw_alm` and `raw_blm` with your own data. For multiple skies or beams, pass a vector of matrices and set `numofsky` / `numofbeam` to the corresponding counts.


In [ ]:
packed_index(l, m, lmax) =
    m * (lmax + 1) - m * (m - 1) ÷ 2 + (l - m) + 1

function toy_coefficients(lmax; beam=false, azimuth_max=lmax)
    values = zeros(ComplexF64, 3, (lmax + 1) * (lmax + 2) ÷ 2)
    for m in 0:min(azimuth_max, lmax), l in m:lmax, row in 1:3
        row > 1 && l < 2 && continue
        phase = 0.31*l + 0.47*m + 0.23*row
        value = complex(cos(phase), m == 0 ? 0.0 : 0.4*sin(phase))
        envelope = beam ? exp(-0.008*l*(l+1)) : 1.0
        values[row, packed_index(l, m, lmax)] =
            envelope * value / (l + 1)^2
    end
    return values
end

raw_alm = toy_coefficients(cp.lmax)
raw_blm = toy_coefficients(cp.lmax; beam=true, azimuth_max=cp.bnmax)

alm = prepare_alm(cp, raw_alm; input_basis=:teb, precision=compute_precision)
blm = prepare_blm(cp, raw_blm; input_basis=:teb, precision=compute_precision)

(raw_alm_size=size(raw_alm), raw_blm_size=size(raw_blm),
 prepared_alm=alm, prepared_blm=blm)


`prepare_alm` and `prepare_blm` convert `(T,E,B)` to the internal `(0,+2,-2)` spin basis.

\[
a_{+2}=-(a_E+i a_B),\qquad a_{-2}=-(a_E-i a_B).
\]

Sky preparation reconstructs negative `m` modes. Beam preparation also applies the complex conjugation needed for convolution; do not conjugate the beam before passing it in. The next cell checks the spin conversion, negative modes, and beam conjugation.

For inputs already in the spin basis, use `input_basis=:spin`.


In [ ]:
let l = 3
    a0, ap, am = alm_slice(alm, l)
    b0, bp, bm = blm_slice(blm, l)
    nmax = calc_nmax(cp, l)
    for m in 1:min(2, cp.bnmax)
        k = packed_index(l, m, cp.lmax)
        @assert a0[1,l+m+1] ≈ Complex{compute_precision}(raw_alm[1,k])
        @assert ap[1,l+m+1] ≈ Complex{compute_precision}(-(raw_alm[2,k] + im*raw_alm[3,k]))
        @assert am[1,l+m+1] ≈ Complex{compute_precision}(-(raw_alm[2,k] - im*raw_alm[3,k]))
        @assert a0[1,l-m+1] == (-1)^m * conj(a0[1,l+m+1])
        @assert ap[1,l-m+1] == (-1)^m * conj(am[1,l+m+1])

        raw_spin = (raw_blm[1,k],
                    -(raw_blm[2,k] + im*raw_blm[3,k]),
                    -(raw_blm[2,k] - im*raw_blm[3,k]))
        for (block, value) in zip((b0, bp, bm), raw_spin)
            @assert block[nmax+m+1,1] ≈ conj(Complex{compute_precision}(value))
        end
        @assert b0[nmax-m+1,1] == (-1)^m * conj(b0[nmax+m+1,1])
        @assert bp[nmax-m+1,1] == (-1)^m * conj(bm[nmax+m+1,1])
    end
end
println("Verified sky/beam spin conversion, negative modes, and conjugation.")


## 4. Define the Falcons.jl scan

Use a single boresight detector, no HWP, and ecliptic coordinates (`coord="E"`). This LiteBIRD-like scan example does not require an instrument model (IMO) for a specific detector.

`alpha` / `beta` are in degrees, rotation rates are in rpm, `get_pointings` times are in seconds, and returned angles are in radians. In Falcons 1.0.1, `get_pointings(ss, start, stop)` returns `(theta, phi, psi, time)`, with angle array axes `(sample, detector)`.

See the [Falcons pointing implementation](https://github.com/yusuke-takase/Falcons.jl/blob/v1.0.1/src/function/scanning.jl) and [angle conventions](https://github.com/yusuke-takase/Falcons.jl/blob/v1.0.1/docs/src/Scanning.md). Pass `psi` directly using Falcons' COSMO/HEALPix convention. This example adds no detector polarization-angle offset or HWP angle. When using a real detector, align its polarization-angle definition and the input sky coordinate system.


In [ ]:
ss = Falcons.gen_ScanningStrategy(
    nside=cp.nside_output,
    duration=mission_seconds,
    sampling_rate=sampling_rate_hz,
    alpha=45.0,
    beta=50.0,
    prec_rpm=Falcons.period2rpm(192.348),
    spin_rpm=0.05,
    hwp_rpm=0.0,
    coord="E",
    start_angle=0.0,
    name=["boresight"],
)

Falcons.show_ss(ss)


## 5. Generate monthly chunks and retain the selected rings

`falcons_chunk` computes one interval, selects the first detector column, and returns `(phi, theta, psi)` in that order. Views avoid copying the angle arrays when selecting the column.

The generator below computes the next month only when `build_ring_scan_buffer` requests it. It does not call `collect(pointing_chunks)` or build a list of arrays for all months. Samples outside the selected rings are not stored in the buffer.

Once an input chunk is no longer referenced, its original monthly arrays become eligible for garbage collection. The retained samples' `phi/theta/psi` remain in the buffer. Construction also needs array-growth and pixel-binning workspace, so the final buffer size does not represent peak memory use.


In [ ]:
function falcons_chunk(ss, start_s, stop_s; detector_index=1)
    theta, phi, psi, times = Falcons.get_pointings(ss, start_s, stop_s)
    @assert size(theta) == size(phi) == size(psi)
    @assert 1 <= detector_index <= size(theta, 2)
    expected = round(Int, (stop_s - start_s) * ss.sampling_rate)
    @assert size(theta, 1) == length(times) == expected
    @assert first(times) ≈ start_s
    @assert last(times) ≈ stop_s - 1/ss.sampling_rate
    return (
        @view(phi[:, detector_index]),
        @view(theta[:, detector_index]),
        @view(psi[:, detector_index]),
    )
end

pointing_chunks = (
    falcons_chunk(ss, start_s, stop_s)
    for (start_s, stop_s) in time_windows
);


In [ ]:
@time buffer = build_ring_scan_buffer(
    cp,
    pointing_chunks;
    rings=target_rings,
    progress_callback=p -> begin
        @printf("chunk %2d/%d: processed=%d, retained=%d\n",
                p.chunk_index, chunk_count, p.processed_samples, p.stored_samples)
        flush(stdout)
    end,
)

@assert buffer.processed_samples == round(Int, mission_seconds * sampling_rate_hz)
@assert buffer.first_ring == first(target_rings)
@assert buffer.last_ring == last(target_rings)
@assert sum(ring_hit_count(buffer, r) for r in target_rings) == buffer.stored_samples
buffer


The scan buffer now covers one year. The original full-year time-ordered data (TOD) do not need to be kept.

`ring_scan_buffer_bytes` reports the total size of the completed angle arrays and pixel offsets. Inspect the hit counts per ring and per retained pixel on the CPU.


In [ ]:
hit_summary = [(ring=r, hits=ring_hit_count(buffer, r)) for r in target_rings]
@printf("retained: %d / %d samples (%.2f%%), buffer: %.2f MiB\n",
        buffer.stored_samples, buffer.processed_samples,
        100 * buffer.stored_samples / buffer.processed_samples,
        ring_scan_buffer_bytes(buffer) / 2^20)
display(hit_summary)

buffer_pixels = buffer.first_pixel:buffer.last_pixel
pixel_hits = diff(buffer.pixel_offsets)
@assert length(pixel_hits) == length(buffer_pixels)
plot(buffer_pixels, pixel_hits;
     xlabel="HEALPix RING pixel (1-based)", ylabel="Hits",
     title="Retained pointing samples", legend=false)


## 6. Convolve on the GPU and write HDF5

The following steps require a GPU. Without one, they skip; there is no CPU convolution fallback.

Use `beam_batch_size=1`, `l_tile=:auto`, and `max_l_tile=8`. With `reclaim_interval=16`, each GPU worker explicitly reclaims memory after every 16 rings, so no periodic reclamation occurs in this three-ring example. Temporary arrays for each ring are still released when that ring finishes.

Each run creates a fresh temporary output directory, so rerunning the cell does not overwrite existing files. Julia deletes the temporary output on exit. To keep it, choose a new output path in an existing persistent directory.

The first GPU run includes compilation and CUDA initialization. For two GPUs, set `device_ids=[0,1]` and launch the kernel with at least as many Julia threads as GPUs. See the [HDF5 reference](../docs/hdf5_output.md#runtime-options) for all options.


In [ ]:
output_path = nothing

if gpu_available
    @assert Threads.nthreads() >= length(device_ids)
    output_dir = mktempdir()
    output_path = convolve_to_hdf5(
        cp, buffer, alm, blm;
        output_path=joinpath(output_dir, "tutorial.h5"),
        rings=target_rings,
        devices=device_ids,
        beam_batch_size=1,
        beta_order=8,
        l_tile=:auto,
        max_l_tile=8,
        memory_fraction=0.7,
        moment_method=:gemm,
        map_precision=compute_precision,
        rcond=1e-2,
        unseen=NaN,
        reclaim_interval=16,
        sky_ids=["toy-sky"],
        beam_ids=["toy-beam"],
        metadata=Dict(
            "pointing_generator" => "Falcons.jl",
            "falcons_version" => string(Base.pkgversion(Falcons)),
            "coordinate_system" => "Ecliptic",
            "mission_seconds" => mission_seconds,
            "sampling_rate_hz" => sampling_rate_hz,
            "chunk_count" => chunk_count,
        ),
        progress_callback=r -> @printf(
            "saved ring %d: %d hits, %d active pixels, %.3f s\n",
            r.ring, r.hit_count, r.active_pixels, r.elapsed_seconds),
    )
    println("HDF5 output: ", output_path)
else
    @info "Skipped GPU convolution. No HDF5 file was created."
end


## 7. Read completion records and output maps

`read_ring_hdf5` returns a CPU array with axes `(sky, beam, stokes, ring_pixel)`, with Stokes components ordered as `I,Q,U`. Use this reader: its axis order differs from directly reading the HDF5 `map` dataset in Julia.

Pixels with no hits, or with insufficient polarization-angle coverage to satisfy the `rcond` threshold, contain `NaN`. The `rings/completed` dataset distinguishes unprocessed rings from completed ones.


In [ ]:
ring_maps = nothing

if output_path !== nothing
    run_record = h5open(output_path, "r") do file
        (state=only(read(file["run_state"])),
         requested=read(file["rings/requested"]),
         completed=read(file["rings/completed"]))
    end
    @assert run_record.state == 1
    @assert findall(==(1), run_record.requested) == collect(target_rings)
    @assert all(==(1), run_record.completed[target_rings])

    ring_maps = Dict(r => read_ring_hdf5(output_path, r) for r in target_rings)
    for r in target_rings
        maps = ring_maps[r]
        @assert size(maps) == (cp.numofsky, cp.numofbeam, 3,
                              ring_geometry(cp, r).pixel_count)
        @printf("ring %d: size=%s, finite=%d / %d\n",
                r, string(size(maps)), count(isfinite, maps), length(maps))
    end
else
    @info "Skipping HDF5 readback because no GPU output is available."
end


In [ ]:
if ring_maps !== nothing
    ring_to_plot = target_rings[cld(length(target_rings), 2)]
    geometry = ring_geometry(cp, ring_to_plot)
    maps = ring_maps[ring_to_plot]
    panels = [
        plot(rad2deg.(geometry.phi_pixels), vec(maps[1,1,s,:]);
             xlabel="Ecliptic longitude [deg]", ylabel=label * " [arbitrary units]",
             legend=false, title="ring $(ring_to_plot)")
        for (s, label) in enumerate(("I", "Q", "U"))
    ]
    plot(panels...; layout=(3,1), size=(800,700))
else
    @info "Skipping Stokes map plots because no GPU output is available."
end


## 8. Switch to one ring or all rings

Change `target_rings` in the configuration cell and rerun from that cell onward.

```julia
target_rings = 16:16             # A single ring
target_rings = 15:17             # A few rings
target_rings = 1:ring_count(cp)  # All rings
```

Omitting `rings` from `build_ring_scan_buffer` retains all rings. Omitting `rings` from `convolve_to_hdf5` processes every ring in the buffer. A buffer built for a subset of rings cannot later supply rings that were not retained.

Selecting all rings increases the number of retained pointings. For one year at 1 Hz with one detector, the three angle arrays alone require about 0.76 GB (decimal), plus construction workspace. Keep this in mind when increasing the sampling rate or detector count.

The displays and assertions in this notebook check inputs and I/O. Establishing GPU numerical accuracy or the physical validity of a beam model requires a separate comparison against a reference. See the [HDF5 reference](../docs/hdf5_output.md) for execution parameters and the output schema.
